# Coherent Scenes Multi-Model — ComfyUI

Tutarlı sahne/hikaye görselleri üret, video oluştur, ses ekle.
WAN 2.2 I2V + FLUX.1 Dev + Qwen Edit + HunyuanVideo Foley.

## Colab Secrets
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel
- `HF_TOKEN` — HuggingFace (FLUX gated)

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

NODES = {
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'ComfyUI_LayerStyle': 'https://github.com/chflame163/ComfyUI_LayerStyle.git',
    'ComfyUI-Easy-Use': 'https://github.com/yolain/ComfyUI-Easy-Use.git',
    'ComfyUI-Frame-Interpolation': 'https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git',
    'ComfyUI-MiniCPM': 'https://github.com/kijai/ComfyUI-MiniCPM.git',
    'ComfyUI-Custom-Scripts': 'https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git',
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-HunyuanVideo-Foley': 'https://github.com/phazei/ComfyUI-HunyuanVideo-Foley.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Ba\u015far\u0131s\u0131z: {e}')

# === WAN 2.2 I2V Diffusion Models (fp16) ===
print('\U0001f4e5 WAN 2.2 I2V 14B (fp16):')
hf_download('Wan-AI/Wan2.2-I2V-14B-720P', 'wan2.2_i2v_high_noise_14B_fp16.safetensors', f'{MODELS_DIR}/diffusion_models')
hf_download('Wan-AI/Wan2.2-I2V-14B-720P', 'wan2.2_i2v_low_noise_14B_fp16.safetensors', f'{MODELS_DIR}/diffusion_models')

# === Text Encoders ===
print('\n\U0001f4e5 Text Encoders:')
hf_download('Kijai/WanVideo_comfy', 'umt5_xxl_fp16.safetensors', f'{MODELS_DIR}/text_encoders')
hf_download('comfyanonymous/flux_text_encoders', 'clip_l.safetensors', f'{MODELS_DIR}/text_encoders')
hf_download('comfyanonymous/flux_text_encoders', 't5xxl_fp16.safetensors', f'{MODELS_DIR}/text_encoders')
hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', f'{MODELS_DIR}/text_encoders')

# === VAE ===
print('\n\U0001f4e5 VAE:')
hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan_2.1_vae.safetensors', f'{MODELS_DIR}/vae')
hf_download('black-forest-labs/FLUX.1-dev', 'ae.safetensors', f'{MODELS_DIR}/vae')
hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/vae/qwen_image_vae.safetensors', f'{MODELS_DIR}/vae')

# === LoRA'lar ===
print('\n\U0001f4e5 LoRA\'lar:')
os.makedirs(f'{MODELS_DIR}/loras/Wan/Wan2.2-I2V-A14B-Moe-Distill-Lightx2v', exist_ok=True)
os.makedirs(f'{MODELS_DIR}/loras/Wan/Wan2.2-Fun-Reward-LoRAs', exist_ok=True)
os.makedirs(f'{MODELS_DIR}/loras/Wan', exist_ok=True)
hf_download('lightx2v/Wan2.2-I2V-A14B-Moe-Distill-Lightx2v', 'high_noise_model_rank64.safetensors', f'{MODELS_DIR}/loras/Wan/Wan2.2-I2V-A14B-Moe-Distill-Lightx2v')
hf_download('lightx2v/Wan2.2-I2V-A14B-Moe-Distill-Lightx2v', 'low_noise_model_rank64.safetensors', f'{MODELS_DIR}/loras/Wan/Wan2.2-I2V-A14B-Moe-Distill-Lightx2v')
hf_download('Wan-AI/Wan2.2-Fun-Reward-LoRAs', 'Wan2.2-Fun-A14B-InP-high-noise-MPS.safetensors', f'{MODELS_DIR}/loras/Wan/Wan2.2-Fun-Reward-LoRAs')
hf_download('Wan-AI/Wan2.2-Fun-Reward-LoRAs', 'Wan2.2-Fun-A14B-InP-low-noise-HPS2.1.safetensors', f'{MODELS_DIR}/loras/Wan/Wan2.2-Fun-Reward-LoRAs')
hf_download('lightx2v/Qwen-Image-Lightning', 'Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors', f'{MODELS_DIR}/loras/Wan')

# === Qwen Image Edit ===
print('\n\U0001f4e5 Qwen Image Edit:')
hf_download('Comfy-Org/Qwen-Image-Edit_ComfyUI', 'split_files/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors', f'{MODELS_DIR}/diffusion_models')

# === FLUX.1 Dev (gated) ===
print('\n\U0001f4e5 FLUX.1 Dev:')
hf_download('black-forest-labs/FLUX.1-dev', 'flux1-dev.safetensors', f'{MODELS_DIR}/diffusion_models')
os.makedirs(f'{MODELS_DIR}/diffusion_models/flux', exist_ok=True)
flux_src = f'{MODELS_DIR}/diffusion_models/flux1-dev.safetensors'
flux_link = f'{MODELS_DIR}/diffusion_models/flux/flux1-dev.sft'
if os.path.exists(flux_src) and not os.path.exists(flux_link):
    os.symlink(flux_src, flux_link)

# === Foley (ses) ===
print('\n\U0001f4e5 Foley Audio:')
os.makedirs(f'{MODELS_DIR}/foley', exist_ok=True)
hf_download('phazei/HunyuanVideo-Foley', 'hunyuanvideo_foley_fp8_e4m3fn.safetensors', f'{MODELS_DIR}/foley')
hf_download('phazei/HunyuanVideo-Foley', 'synchformer_state_dict_fp16.safetensors', f'{MODELS_DIR}/foley')
hf_download('phazei/HunyuanVideo-Foley', 'vae_128d_48k_fp16.safetensors', f'{MODELS_DIR}/foley')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy
`USE_CLOUDFLARE = True`: Cloudflare tunnel

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False
PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI ba\u015flat\u0131ld\u0131 (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI \u00e7\u00f6kt\u00fc!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI haz\u0131r ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canl\u0131 tutma. Durdurmak i\u00e7in interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)